In [ ]:
import os
import sys
import os

ROOT_DIR = "ABSOLUTE_PATH_TO_PROJECT_ROOT"
sys.path.append(os.path.join(ROOT_DIR, "code"))

import json
import math
import multiprocessing
from functools import partial
from collections import defaultdict

import numpy as np
import pandas as pd

from matplotlib import pyplot as plt

from pprint import pprint
from tqdm.auto import tqdm

from models.uncertainty.dist_match.tree import DistMatchQRF
from models.uncertainty.dist_match.utils import match_ks_stat, match_ks_p_val

In [3]:
MATCH_THRESHOLD = 0.6


def matcher(x1, x2):
    return match_ks_stat(x1, x2) < MATCH_THRESHOLD


def get_qrf(path: str) -> DistMatchQRF:
    qrf = DistMatchQRF(
        alpha=0.1,
        n_quantile_bins=10,
        feature_dim=-1,
        matcher=matcher,
        match_mask=None,
        n_trees=10,
        bagging_ratio=0.9,
        verbose=False,
    )
    qrf.load_trees(path)
    return qrf

In [ ]:
def get_file_dir(data_key: str, table_key: str) -> str:
    return os.path.join(ROOT_DIR, f"outputs/{data_key}/wandb/latest-run/files/media/table/Eval_{table_key}_plots")


def read_json(path: str) -> pd.DataFrame:
    with open(path, "r") as file:
        data = json.load(file)
    cols = data["columns"]
    data = data["data"]
    data = [dict(zip(cols, datum)) for datum in data]
    return pd.DataFrame.from_records(data)


def get_data_df(data_key: str, table_key: str, file_idx: int = 0) -> pd.DataFrame:
    file_dir = get_file_dir(data_key, table_key)
    filename = os.listdir(file_dir)[file_idx]
    return read_json(os.path.join(file_dir, filename))

In [5]:
def max_len_from_same_distro(values):
    max_len = 0
    n_values = len(values)
    for val1 in values:
        cur_len = 0
        for val2 in values:
            cur_len += int(match_ks_p_val(val1, val2).item() > 0.05)

        max_len = max(cur_len, max_len)
        if max_len == n_values:
            break
    return max_len


def study_qrf_ood_samples(data, qrf, patch_len: int = 100):
    n_values_ratio = 0
    n_leaves = 0
    n_ood_values = 0

    calib_ids = data["step"] >= 0
    calib_len = len(calib_ids)

    n_trees = len(qrf.trees)

    for tree in qrf.trees:
        node = tree.leaf_nodes[0] # 0 -- leftmost node
        values = node.get_values()[0]
        max_len = max_len_from_same_distro(values)
        n_values = len(values)

        print(f"{max_len} / {n_values}")
        n_values_ratio += max_len / n_values
        n_leaves += len(tree.leaf_nodes)
        n_ood_values += n_values - max_len
    
    n_leaves /= n_trees
    n_values_ratio /= n_trees
    n_ood_values /= n_trees
    return n_values_ratio, n_leaves, calib_len, n_ood_values

In [ ]:
PATCH_LEN = 100
TREE_DIR = os.path.join(ROOT_DIR, "models_save/uc/")
DATA_MAP = {
    "Elec": (
        # "qrf_ks_stat<0.1_error_normal_electricelectricity-normalized_darts-forest|100.pkl",
        "qrf_ks_stat<0.01_error_normal_electricelectricity-normalized_darts-forest|100.pkl",
        #"darts_forest_dist_match_enbPI_electric_s20_260126_140743",
        "darts_forest_dist_match_enbPI_electric_s20_270126_170906",
        "electricelectricity-normalized",
        0,
    ),
    "Solar": (
        "qrf_ks_error_normal_solarSolar_Atl_data_aligned_darts-forest|100.pkl",
        "darts_forest_dist_match_enbPI_solar_atlanta_280525_144732",
        "solarSolar_Atl_data_aligned",
        1,
    ),
    "Wind": (
        "qrf_ks_error_normal_windWind_Hackberry_Generation_2019_2020_darts-forest|100.pkl",
        "darts_forest_dist_match_enbPI_wind_s30_070725_162757",
        "windWind_Hackberry_Generation_2019_2020",
        0,
    ),
    "META": (
        "qrf_ks_error_normal_stockMETA_darts-forest|100.pkl",
        "darts_forest_dist_match_stock_meta_s70_170725_142303",
        "stockMETA",
        0,
    ),
    "NFLX": (
        "qrf_ks_error_normal_stockNFLX_darts-forest|100.pkl",
        "darts_forest_dist_match_stock_nflx_s30_170725_142303",
        "stockNFLX",
        0,
    ),
    "PEMS": (
        "qrf_ks_error_normal_data_darts-forest|100.pkl",
        "darts_forest_dist_match_pems_s20_180725_164234",
        "pems2022_01",
        0
    ),
    "rain": (
        "qrf_ks_error_normal_raindaily_weather_darts-forest|100.pkl",
        "darts_forest_dist_match_rain_s50_040825_124049",
        "raindaily_weather",
        0
    )
}

for data_type, (qrf_path, data_key, table_key, file_idx) in DATA_MAP.items():
    qrf = get_qrf(os.path.join(TREE_DIR, qrf_path))
    data = get_data_df(data_key, table_key, file_idx)
    n_values_ratio, n_leaves, calib_len, n_ood_values = study_qrf_ood_samples(
        data, qrf, PATCH_LEN
    )
    print(f"[{data_type}] same_distro: {n_values_ratio:.2}; n_leaves: {n_leaves}; calib_len: {calib_len}; n_ood: {n_ood_values}; ood_ratio: {n_ood_values / calib_len:.2f}")

169 / 252
166 / 247
154 / 236
163 / 257
160 / 240
163 / 235
162 / 250
168 / 249
160 / 238
160 / 246
[Elec] same_distro: 0.66; n_leaves: 245.0; calib_len: 1378; n_ood: 82.5; ood_ratio: 0.06
62 / 75
61 / 69
55 / 70
55 / 67
62 / 74
56 / 69
59 / 74
59 / 73
53 / 68
48 / 65
[Solar] same_distro: 0.81; n_leaves: 70.4; calib_len: 800; n_ood: 13.4; ood_ratio: 0.02
364 / 721
366 / 721
369 / 720
364 / 709
347 / 708
351 / 735
356 / 729
351 / 689
354 / 718
